# 2. Go2 のバク転の学習：最初の報酬設計（v1）

Go2 にバク転を学習させる最初の報酬設計。**この報酬では 1/4 回転して元の姿勢に戻る動きに落ち着き、バク転はできなかった。**
改善した報酬は `03_go2_backflip_v2.ipynb` にある。

- 学習・評価・動画作成まですべて Google Colab 上で行い、ローカルには何もインストールしない
- 事前に VS Code の Colab 拡張で **GPU（T4 など）ランタイム** に接続しておく
- ランタイムが切れると環境は消えるので、毎回上から実行し直す
- このノートブックだけで完結する（ほかの 2 つのノートブックを先に実行する必要はない）

In [ ]:
# GPU が割り当てられているか確認
!nvidia-smi

## 1. インストール

「セッションを再起動」を求められたら、カーネルを再起動してから次のセルに進む。

In [ ]:
!pip install -q genesis-world

In [ ]:
import importlib.metadata as md
import os
import re

GS_VER = md.version('genesis-world')
print('genesis-world', GS_VER)

# インストールした Genesis と同じバージョンのサンプルコードを取得（タグがなければ最新版）
GENESIS_DIR = '/content/Genesis'
if not os.path.exists(GENESIS_DIR):
    repo = 'https://github.com/Genesis-Embodied-AI/Genesis.git'
    if os.system(f'git clone -q --depth 1 --branch v{GS_VER} {repo} {GENESIS_DIR}') != 0:
        os.system(f'git clone -q --depth 1 {repo} {GENESIS_DIR}')
LOCO_DIR = f'{GENESIS_DIR}/examples/locomotion'

# go2_train.py が要求する rsl_rl のバージョン（例: >=5.0.0）を読み取ってインストール
m = re.search(r"rsl-rl-lib\s*([<>=!~]=?\s*[0-9.]+)", open(f'{LOCO_DIR}/go2_train.py').read())
RSL_REQ = f'rsl-rl-lib{m.group(1)}' if m else 'rsl-rl-lib'
print('install', RSL_REQ)
!pip install -q "{RSL_REQ}" tensordict tensorboard

In [ ]:
# Colab では EGL が NVIDIA GPU を見つけるための設定ファイルがなく、描画の初期化で Segmentation fault になる
# （MuJoCo の公式 Colab チュートリアルと同じ対処）
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
os.environ['PYOPENGL_PLATFORM'] = 'egl'  # 学習スクリプト（子プロセス）にも引き継がれる
!ldconfig -p | grep -i egl_nvidia

## 2. Google Drive（任意）

学習結果を残したい場合のみ `USE_DRIVE = True` にする。

In [ ]:
USE_DRIVE = True
SAVE_DIR = None

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/pai_finalwork'
    os.makedirs(SAVE_DIR, exist_ok=True)

## 3. 報酬と学習スクリプトを作る

公式の `go2_backflip.py` には学習済みモデル（お手本）しか入っておらず、学習コードがないので、報酬と学習スクリプトを自作する。

- 元のファイル（`go2_env.py` / `go2_train.py` / `go2_backflip.py`）は書き換えず、`%%writefile` で新しいファイルを作る
- 報酬は `go2_backflip_v1_env.py` に書く（v1 と v2 でファイル名を分けているので、同じランタイムで両方を実行しても混ざらない）
- 学習と録画のスクリプト（`go2_backflip_train.py` / `go2_backflip_record.py`）は v1・v2 で共通
- 学習も評価も別プロセス（`!python`）で動かすので、`gs.init()` の再起動問題は起きない

| 時間 | 段階 | 報酬 |
|---|---|---|
| 0〜0.5 秒 | 踏み切り | 上向きの速度（`jump`） |
| 0.5〜1.1 秒 | 空中で回転 | 後ろ向きの回転速度（`ang_vel_y`） |
| 1.1〜2.0 秒 | 着地 | 立ち姿勢の高さにいるか（`landing_height`） |
| 全体 | | 時刻ごとの目標の姿勢に近いこと（`orientation_tracking`） |

回りきれずに元に戻っても、着地の点数は満点になる。そのため「挑戦して転ぶより、少し回って戻るほうが得」になってしまった。

In [ ]:
%%writefile {LOCO_DIR}/go2_backflip_v1_env.py
import math

import torch

from go2_backflip import get_cfgs as get_backflip_cfgs
from go2_env import Go2Env


def get_cfgs():
    # 環境の設定は公式の go2_backflip.py のもの（PD ゲイン、傾きで打ち切らない など）を流用する
    env_cfg, obs_cfg, _, command_cfg = get_backflip_cfgs()
    env_cfg["episode_length_s"] = 2.0
    reward_cfg = {
        "flip_start_s": 0.5,  # 回転を始める時刻
        "flip_duration_s": 0.6,  # 1 回転にかける時間
        "orientation_sigma": 0.25,
        "max_ang_vel_y": 10.0,  # これ以上速く回っても報酬は増えない [rad/s]
        "base_height_target": 0.3,
        "reward_scales": {
            "orientation_tracking": 5.0,
            "ang_vel_y": 1.0,
            "jump": 2.0,
            "landing_height": 2.0,
            "ang_vel_xz": -0.1,
            "dof_vel": -1e-4,
            "action_rate": -0.01,
        },
    }
    return env_cfg, obs_cfg, reward_cfg, command_cfg


class Go2BackflipEnv(Go2Env):
    def _update_observation(self):
        # 公式の BackflipEnv と同じく、エピソード内の経過時間（phase）を観測に加えて 60 次元にする
        phase = torch.pi * self.episode_length_buf[:, None] / self.max_episode_length
        self.obs_buf = torch.cat(
            [
                self.base_ang_vel * self.obs_scales["ang_vel"],  # 3
                self.projected_gravity,  # 3
                (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],  # 12
                self.dof_vel * self.obs_scales["dof_vel"],  # 12
                self.actions,  # 12
                self.last_actions,  # 12
                torch.sin(phase),
                torch.cos(phase),
                torch.sin(phase / 2),
                torch.cos(phase / 2),
                torch.sin(phase / 4),
                torch.cos(phase / 4),
            ],
            dim=-1,
        )

    def _time(self):
        return self.episode_length_buf * self.dt

    def _flip_progress(self):
        # 回転の進み具合（回転前は 0、回転後は 1）
        cfg = self.reward_cfg
        return torch.clamp((self._time() - cfg["flip_start_s"]) / cfg["flip_duration_s"], 0.0, 1.0)

    def _in_flip(self):
        cfg = self.reward_cfg
        t = self._time()
        return ((t >= cfg["flip_start_s"]) & (t < cfg["flip_start_s"] + cfg["flip_duration_s"])).float()

    # Reward functions.
    def _reward_orientation_tracking(self):
        # 時刻ごとの目標の姿勢に近いほど高い。
        # バク転は頭が上がる向き＝体の y 軸まわりに負の回転なので、目標のピッチ角は 0 → -2π
        pitch = -2 * math.pi * self._flip_progress()
        target_gravity = torch.stack([torch.sin(pitch), torch.zeros_like(pitch), -torch.cos(pitch)], dim=-1)
        error = torch.sum(torch.square(self.projected_gravity - target_gravity), dim=1)
        return torch.exp(-error / self.reward_cfg["orientation_sigma"])

    def _reward_ang_vel_y(self):
        # 回転中は後ろ向きの回転速度を評価
        return torch.clamp(-self.base_ang_vel[:, 1], max=self.reward_cfg["max_ang_vel_y"]) * self._in_flip()

    def _reward_jump(self):
        # 踏み切り直前から回転の前半までは、上向きの速度（ワールド座標）を評価
        cfg = self.reward_cfg
        t = self._time()
        active = (t >= cfg["flip_start_s"] - 0.1) & (t < cfg["flip_start_s"] + 0.5 * cfg["flip_duration_s"])
        return torch.clamp(self.robot.get_vel()[:, 2], min=0.0, max=3.0) * active.float()

    def _reward_landing_height(self):
        # 着地後は、立ち姿勢の高さに近いほど高い
        cfg = self.reward_cfg
        landed = self._time() >= cfg["flip_start_s"] + cfg["flip_duration_s"] + 0.2
        height_error = torch.square(self.base_pos[:, 2] - cfg["base_height_target"])
        return torch.exp(-height_error / 0.01) * landed.float()

    def _reward_ang_vel_xz(self):
        # 横回転・ひねりを抑える（ペナルティ）
        return torch.sum(torch.square(self.base_ang_vel[:, [0, 2]]), dim=1)

    def _reward_dof_vel(self):
        # 関節の速すぎる動きを抑える（ペナルティ）
        return torch.sum(torch.square(self.dof_vel), dim=1)

In [ ]:
%%writefile {LOCO_DIR}/go2_backflip_train.py
import argparse
import importlib
import os
import pickle
import shutil

from rsl_rl.runners import OnPolicyRunner

import genesis as gs

from go2_train import get_train_cfg


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-backflip")
    parser.add_argument("-b", "--num-envs", type=int, default=4096)
    parser.add_argument("--max-iterations", type=int, default=1000)
    parser.add_argument("--seed", type=int, default=1)
    parser.add_argument("--env", type=str, required=True, help="報酬を定義したモジュール名（例: go2_backflip_v1_env）")
    args = parser.parse_args()

    env_module = importlib.import_module(args.env)
    log_dir = f"logs/{args.exp_name}"
    env_cfg, obs_cfg, reward_cfg, command_cfg = env_module.get_cfgs()
    train_cfg = get_train_cfg(args.exp_name)

    if os.path.exists(log_dir):
        shutil.rmtree(log_dir)
    os.makedirs(log_dir, exist_ok=True)

    with open(f"{log_dir}/cfgs.pkl", "wb") as f:
        pickle.dump([env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg], f)
    # 録画のときに同じ環境を使えるよう、モジュール名も残しておく
    with open(f"{log_dir}/env_module.txt", "w") as f:
        f.write(args.env)

    gs.init(backend=gs.gpu, precision="32", logging_level="warning", seed=args.seed, performance_mode=True)

    env = env_module.Go2BackflipEnv(
        num_envs=args.num_envs, env_cfg=env_cfg, obs_cfg=obs_cfg, reward_cfg=reward_cfg, command_cfg=command_cfg
    )

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)

    # 歩行と違い、バク転は毎回エピソードの最初（踏み切り前）から始める必要があるので、開始時刻をばらつかせない
    runner.learn(num_learning_iterations=args.max_iterations, init_at_random_ep_len=False)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {LOCO_DIR}/go2_backflip_record.py
import argparse
import glob
import importlib
import os
import pickle
import re

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import torch

import genesis as gs


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("-e", "--exp-name", type=str, default="go2-backflip")
    parser.add_argument("--ckpt", type=int, default=None, help="使うモデルの番号（model_<番号>.pt）。省略すると一番新しいもの")
    parser.add_argument(
        "--ckpts", type=int, nargs="+", default=None,
        help="複数のモデルをまとめて録画する（例: --ckpts 0 100 200）。動画は -o のフォルダに model_<番号>.mp4 で保存",
    )
    parser.add_argument(
        "--env", type=str, default=None,
        help="報酬を定義したモジュール名。省略すると学習のときと同じもの（logs/<実験名>/env_module.txt）",
    )
    parser.add_argument("--pretrained", action="store_true", help="公式の学習済みモデル（お手本）を使う")
    parser.add_argument("--episodes", type=int, default=3, help="録画するバク転の回数")
    parser.add_argument("-o", "--output", type=str, default="/content/backflip.mp4")
    args = parser.parse_args()

    gs.init(backend=gs.gpu, logging_level="warning")

    if args.pretrained:
        from go2_backflip import BackflipEnv, get_cfgs

        env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()
        env_cfg["episode_length_s"] = 2
        env_cls = BackflipEnv
    else:
        log_dir = f"logs/{args.exp_name}"
        # 学習のときに使った報酬のモジュール（学習スクリプトが env_module.txt に書き残している）
        env_module = args.env or open(f"{log_dir}/env_module.txt").read().strip()
        Go2BackflipEnv = importlib.import_module(env_module).Go2BackflipEnv
        env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"{log_dir}/cfgs.pkl", "rb"))
        reward_cfg["reward_scales"] = {}
        env_cls = Go2BackflipEnv

    # 環境は内部でシーンを build してしまうので、build の直前に横から見るカメラを追加する
    cams = []
    orig_build = gs.Scene.build

    def build_with_camera(self, *build_args, **build_kwargs):
        cams.append(self.add_camera(res=(640, 480), pos=(0.0, 2.5, 0.6), lookat=(0.0, 0.0, 0.4), fov=40, GUI=False))
        return orig_build(self, *build_args, **build_kwargs)

    gs.Scene.build = build_with_camera
    env = env_cls(
        num_envs=1,
        env_cfg=env_cfg,
        obs_cfg=obs_cfg,
        reward_cfg=reward_cfg,
        command_cfg=command_cfg,
        show_viewer=False,
    )
    gs.Scene.build = orig_build
    cam = cams[0]

    def record(policy, output):
        obs = env.reset()
        cam.start_recording(save_to_filename=output, fps=int(1 / env.dt))
        with torch.no_grad():
            # エピソードは max_episode_length ステップを超えると自動でリセットされ、次のバク転が始まる
            for _ in range((env.max_episode_length + 1) * args.episodes):
                obs, _, _, _ = env.step(policy(obs))
        cam.stop_recording()
        print("saved", output)

    if args.pretrained:
        # go2_backflip.py と同じネットワークを作って重みを読み込む
        policy = torch.nn.Sequential(
            torch.nn.Linear(obs_cfg["num_obs"], 512),
            torch.nn.ELU(),
            torch.nn.Linear(512, 256),
            torch.nn.ELU(),
            torch.nn.Linear(256, 128),
            torch.nn.ELU(),
            torch.nn.Linear(128, env_cfg["num_actions"]),
        )
        policy.load_state_dict(torch.load("backflip/single.pt", weights_only=True))
        policy.to(device=gs.device)
        record(policy, args.output)
        return

    from rsl_rl.runners import OnPolicyRunner

    runner = OnPolicyRunner(env, train_cfg, log_dir, device=gs.device)

    def load_policy(ckpt_path):
        print("load", ckpt_path)
        runner.load(ckpt_path)
        return runner.get_inference_policy(device=gs.device)

    if args.ckpts is not None:
        # 環境は 1 回だけ作り、モデルを差し替えながら録画する
        os.makedirs(args.output, exist_ok=True)
        for ckpt in args.ckpts:
            record(load_policy(f"{log_dir}/model_{ckpt}.pt"), f"{args.output}/model_{ckpt}.mp4")
        return

    if args.ckpt is None:
        ckpt_path = max(glob.glob(f"{log_dir}/model_*.pt"), key=lambda p: int(re.findall(r"model_([0-9]+)", p)[0]))
    else:
        ckpt_path = f"{log_dir}/model_{args.ckpt}.pt"
    record(load_policy(ckpt_path), args.output)


if __name__ == "__main__":
    main()

## 4. お手本（公式の学習済みモデル）を見る（任意）

In [ ]:
from IPython.display import Video

%cd {LOCO_DIR}
!python go2_backflip_record.py --pretrained -o /content/backflip_pretrained.mp4
Video('/content/backflip_pretrained.mp4', embed=True, width=640)

## 5. 学習

1000 回、5001 回と学習を進めても、高くジャンプするようにはなったが 1/4 回転して元に戻る動きのままだった。

In [ ]:
import shutil

BF_EXP_NAME = 'go2-backflip-v1'   # 設定を変えて学習するときは名前も変える（同じ名前だと前の結果が消える）
BF_NUM_ENVS = 4096
BF_MAX_ITER = 5001   # モデルは 100 回ごと（model_0, model_100, ...）に保存される。5001 にすると model_5000 まで残る

%cd {LOCO_DIR}
!python -X faulthandler go2_backflip_train.py --env go2_backflip_v1_env -e {BF_EXP_NAME} -b {BF_NUM_ENVS} --max-iterations {BF_MAX_ITER}
!ls {LOCO_DIR}/logs/{BF_EXP_NAME}

if SAVE_DIR:
    shutil.copytree(f'{LOCO_DIR}/logs/{BF_EXP_NAME}', f'{SAVE_DIR}/logs/{BF_EXP_NAME}', dirs_exist_ok=True)
    print('saved to', f'{SAVE_DIR}/logs/{BF_EXP_NAME}')

## 6. 評価（学習したモデルでバク転を録画）

### 6-1. 1 つのモデルを録画する

In [ ]:
import shutil

from IPython.display import Video

EVAL_EXP_NAME = BF_EXP_NAME   # 別の実験の結果を見るときは名前を書き換える
BF_CKPT = None                # 途中のモデルを使うときは番号を指定する（例: 100 → model_100.pt）。None なら一番新しいもの

ckpt_opt = '' if BF_CKPT is None else f'--ckpt {BF_CKPT}'
BF_VIDEO_PATH = f'/content/{EVAL_EXP_NAME}' + ('' if BF_CKPT is None else f'_{BF_CKPT}') + '.mp4'

%cd {LOCO_DIR}
!ls {LOCO_DIR}/logs/{EVAL_EXP_NAME}
!python go2_backflip_record.py -e {EVAL_EXP_NAME} {ckpt_opt} -o {BF_VIDEO_PATH}
if SAVE_DIR:
    shutil.copy(BF_VIDEO_PATH, SAVE_DIR)
Video(BF_VIDEO_PATH, embed=True, width=640)

### 6-2. 学習の途中のモデルをまとめて録画し、過程を見比べる

In [ ]:
# 保存されているモデルを一定の間隔でまとめて録画し、学習の過程を見比べる
import glob
import shutil

from IPython.display import HTML, Video, display

EVAL_EXP_NAME = BF_EXP_NAME   # 別の実験の結果を見るときは名前を書き換える
CKPT_STEP = 100               # 何回ごとのモデルを録画するか（100 の倍数。500 にすると本数が 1/5 になる）
EPISODES = 3                  # モデルごとに録画するバク転の回数
SHOW_VIDEOS = False            # ノートブックに動画を表示するか（本数が多いと ipynb が重くなる）

log_dir = f'{LOCO_DIR}/logs/{EVAL_EXP_NAME}'
ckpts = sorted(int(re.findall(r'model_([0-9]+)', p)[0]) for p in glob.glob(f'{log_dir}/model_*.pt'))
ckpts = [c for c in ckpts if c % CKPT_STEP == 0 or c == ckpts[-1]]
ckpt_args = ' '.join(map(str, ckpts))
print('録画するモデル:', ckpts)

VIDEO_DIR = f'/content/videos/{EVAL_EXP_NAME}'
%cd {LOCO_DIR}
!python go2_backflip_record.py -e {EVAL_EXP_NAME} --ckpts {ckpt_args} --episodes {EPISODES} -o {VIDEO_DIR}

if SAVE_DIR:
    shutil.copytree(VIDEO_DIR, f'{SAVE_DIR}/videos/{EVAL_EXP_NAME}', dirs_exist_ok=True)
    print('saved to', f'{SAVE_DIR}/videos/{EVAL_EXP_NAME}')

if SHOW_VIDEOS:
    for c in ckpts:
        display(HTML(f'<b>model_{c}</b>'))
        display(Video(f'{VIDEO_DIR}/model_{c}.mp4', embed=True, width=480))